# VAE Assignment — KL Divergence, Reparameterization, ELBO

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence

torch.manual_seed(0)

## Part 1 — Closed-Form KL, From Scratch (30 pts)

In [ ]:
def kl_divergence_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    var = torch.exp(logvar)
    kl = 0.5 * (mu.pow(2) + var - logvar - 1)
    return kl.sum()

mu     = torch.randn(32, 16)
logvar = torch.randn(32, 16)
sigma  = torch.exp(0.5 * logvar)

your_kl = kl_divergence_gaussian(mu, logvar)

q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(sigma))
lib_kl  = kl_divergence(q, p).sum()

print(f"Your KL:   {your_kl:.4f}")
print(f"Torch KL:  {lib_kl:.4f}")
assert torch.isclose(your_kl, lib_kl, atol=1e-4), "KL values don't match!"
print("Part 1 passed")

## Part 2 — Reparameterization From Scratch (20 pts)

In [ ]:
def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + std * eps

mu_test     = torch.tensor([2.0, -1.0])
logvar_test = torch.tensor([0.0,  0.5])

samples = torch.stack([reparameterize(mu_test, logvar_test) for _ in range(10000)])
print(f"Target mu:       {mu_test.tolist()}")
print(f"Sample mean:     {samples.mean(0).tolist()}")
print(f"Target std:      {torch.exp(0.5 * logvar_test).tolist()}")
print(f"Sample std:      {samples.std(0).tolist()}")
print("Part 2 passed")

## Part 3 — ELBO Loss

In [ ]:
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor) -> torch.Tensor:
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction="sum")
    kl_loss = kl_divergence_gaussian(mu, logvar)
    return recon_loss + kl_loss

batch, dim, latent = 16, 784, 32
x      = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu     = torch.randn(batch, latent)
logvar = torch.randn(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")
assert loss.ndim == 0, "Loss must be a scalar!"
print("Part 3 passed")

## Part 4 — Analysis & Reflection (20 pts)

In [ ]:
mu_zero = torch.zeros(16, 32)
logvar_zero = torch.zeros(16, 32)
kl_zero = kl_divergence_gaussian(mu_zero, logvar_zero)
print(f"KL with mu=0, logvar=0: {kl_zero.item():.4f}")

**1. Why can't we just maximise `log p(x)` directly?**

`log p(x) = log ∫ p(x|z) p(z) dz` requires integrating over every possible value of the latent variable `z`, and for any reasonably expressive decoder this integral has no closed form and is too high-dimensional to evaluate numerically. The encoder `q(z|x)` sidesteps this by proposing a tractable distribution over the `z`'s that are actually likely to have generated `x`, which lets us derive a lower bound (the ELBO) that only needs expectations under `q(z|x)` rather than the full integral.

**2. What would happen if you removed the KL term?**

Without the KL term, the loss only rewards reconstruction accuracy, so the encoder is free to push the `mu` and `logvar` it outputs to whatever values make reconstruction easiest, typically collapsing `logvar` toward very negative values (near-zero variance) and spreading `mu` far apart for different inputs. The latent space stops resembling a standard normal, sampling from `N(0, I)` at generation time would produce points the decoder never saw during training, and the model effectively turns into a deterministic autoencoder with no meaningful generative capability.

**3. Why does the reparameterization trick work?**

Writing `z = mu + std * eps` with `eps ~ N(0, I)` moves the randomness into `eps`, which does not depend on the network parameters, so `z` becomes a deterministic, differentiable function of `mu` and `logvar`. Gradients can then flow from the reconstruction loss through `z` back into `mu` and `std` (and hence into the encoder weights) via ordinary backpropagation. Without this trick, sampling `z` directly from `N(mu, sigma^2)` makes `z` the output of a stochastic node, and there is no well-defined gradient of a sampling operation with respect to its own distribution's parameters, so backpropagation cannot pass through it.

**4. KL with mu=0, logvar=0**

The closed-form KL per dimension is `0.5 * (mu^2 + exp(logvar) - logvar - 1)`. Substituting `mu=0` and `logvar=0` gives `0.5 * (0 + 1 - 0 - 1) = 0`, so the total KL summed over all dimensions and the batch should also be exactly 0. This matches the closed-form formula because `mu=0, logvar=0` means `q(z|x) = N(0, 1)`, which is identical to the prior `p(z) = N(0, 1)`, and the KL divergence between two identical distributions is always 0.